# T5-base — DIMER E2E text-to-text fine-tuning tutorial: teaching a new task prefix (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/t5-base-text2text-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/t5-base-text2text-pipeline/blob/main/tutorials/t5_base_text2text_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google--t5%2Ft5--base-ffcc4d?style=flat)](https://huggingface.co/google-t5/t5-base) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Ftext--to--text--transfer--transformer-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/text-to-text-transfer-transformer) [![arXiv](https://img.shields.io/badge/arXiv-1910.10683-b31b1b.svg)](https://arxiv.org/abs/1910.10683)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** caller-prefixed text-to-text generation (summarisation and English→German/French/Romanian translation) and bounded supervised fine-tuning of the last decoder blocks that teaches a new task prefix on a referenced corpus, using the pinned `google-t5/t5-base` weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/t5_base_text2text_pipeline/`, at revision `5d97bca80c1c`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `a9723ea7f1b39c1eae772870f3b547bf6ef7e6c1` (~894 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned T5-base snapshot (safetensors, 892 MB), fetches the three digest-pinned SciTLDR-A files from the project repository (5.5 MB, no credential), pairs abstracts with their paper titles and draws 300 / 50 / 100 training, validation and test records from the release's own paper-disjoint members, generates from two prefixed inputs through the inference contract with an input manifest and a rejection probe, scores the frozen model on the test abstracts under the new prefix `paper title: ` with ROUGE-1/2/L beside the Lead-1 baseline and the trained `summarize: ` prefix, runs a bounded fine-tuning of the last four decoder blocks that teaches the prefix with validation-ROUGE-L epoch selection, scores the held-out split again, generates titles for new abstracts with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify output parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about eight minutes of model time after the downloads; a CUDA runtime is used automatically when present.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own input–output pairs as a CSV (columns `id`, `source`, `target`), a JSON array or a JSONL file of `{{id, source, target}}` or `{{id, source, targets: [...]}}` records — sources **without** a prefix; set `PREFIX` to the task name you want to teach. They pass through the same validation, seeded source-disjoint split, baselines, fine-tuning, held-out evaluation, inference, artifact export and reload-parity cells as the SciTLDR sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

T5 casts every task as text-to-text: the caller writes a **task prefix** (`summarize: `, `translate English to German: `) in front of the input, the 223 M-parameter encoder-decoder reads the prefixed text once, and the decoder writes the output token by token under **greedy decoding** (`num_beams` 1, the default) or beam search (`num_beams` up to 8). The pinned checkpoint knows the four prefixes in `TASK_PREFIXES`; the carried module never prepends one, reports which known prefix an input starts with (`known_prefix`, `None` otherwise), verifies the snapshot manifest, validates inputs and settings against named ceilings (an input over `MAX_INPUT_TOKENS` is rejected, not truncated), and returns a fixed output contract with `generated_tokens`, `input_tokens` and `stopped_by`. **The pipeline emits no score, probability or quality metric** — a generation is free text.

What this notebook adds to inference is **adaptation to a new prefix**. The dataset is real: SciTLDR-A (Cachola et al., 2020; Apache-2.0) ships, for 3,229 computer-science papers, the abstract and the paper's title — three digest-pinned JSON-Lines files fetched from the project repository at a pinned commit. The tutorial pairs each abstract with its title under the prefix `paper title: `, which the pinned checkpoint has **never seen**: the frozen model answers it with whatever its trained tasks make of the words (the build record saw `True` and `False`, scoring ROUGE-L 0.7), and the fine-tuning question is whether a bounded adaptation of the last decoder blocks teaches the prefix on held-out papers. Three metrics are implemented in the carried `metrics.py` (corpus **ROUGE-1/2/L** F1, rouge-score-style, not rouge-score-identical) and two reference points frame the result: the **Lead-1 baseline** (the abstract's first sentence as the title) and the frozen model's own **`summarize: ` prefix** scored against the titles. Nothing here is a quality claim about your task: it is one seeded split of one corpus.

**Learning objectives:** install the pinned runtime; read what the carried pipeline, dataset and metrics modules guarantee; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned referenced corpus and validate and split it without leakage; generate through the public API with explicit `max_new_tokens`/`num_beams` and read `known_prefix`, the token counts and `stopped_by` correctly; score the frozen model under a new prefix beside the Lead-1 baseline and a trained prefix and read why an unknown prefix means nothing until it is taught; run a bounded fine-tuning with explicit hyperparameters and validation-based epoch selection; evaluate on an independent test split; generate for new abstracts; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** sampling-based or diverse decoding, tasks the checkpoint was not trained on before they are taught here, multi-document or long-document (chunked) generation, full-model or encoder fine-tuning, classification or scoring (the sibling `bart-mnli-zero-shot-classification-pipeline` covers zero-shot classification), any faithfulness or factuality score, and any claim that a SciTLDR title split stands in for your task. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available. CPU is adequate: the build record measured 5.5 s to load and digest-verify the 892 MB snapshot, about 0.3–0.6 s per abstract for 4-beam title-length generation (26–64 s for the 100-abstract test split) and about 100 s per training epoch over 300 abstracts plus a 50-abstract validation pass per epoch. The pinned `torch==2.14.0` install and the 892 MB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python; what an encoder-decoder (seq2seq) model is; what a task prefix does in T5 and why an untrained prefix is just words; what greedy and beam decoding do; what ROUGE measures and why it is neither faithfulness nor a human judgement.
- **Data contract:** records are `{{id, source, targets}}` — an input **without** any prefix and one or more reference outputs (`{{id, source, target}}` with a single string is accepted and normalised), the source 1..20,000 characters and, with the prefix, at most 512 BPE tokens at inference, each reference 1..400 characters, ids matching `[A-Za-z0-9_.:-]{{1,64}}` and unique; a dataset needs 8..20,000 records; sources are de-duplicated case-insensitively before splitting so the same input never sits in two splits; the prefix is a short string ending in `: ` (at most 64 characters); during training only, prefixed sources are truncated to 512 and targets to 64 BPE tokens (inference never truncates — it rejects). BYOD accepts CSV, JSON or JSONL in that shape.
- **Validation is structural, not semantic:** nothing checks that a reference is a good output for its source or that the prefix describes the task — a mislabelled corpus is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — an internal document set with its reference outputs is exactly that. The default path uploads nothing.
- **External access (data):** besides the Hub, the default path fetches three pinned objects (`train.jsonl` 3,155,015 bytes, `dev.jsonl` 1,124,865 bytes, `test.jsonl` 1,204,107 bytes; SHA-256 `b222771d…` / `3191fa98…` / `fb42dd6c…`) from `raw.githubusercontent.com` at the pinned `allenai/scitldr` commit over HTTPS, each refused on any mismatch before it is read; SciTLDR is Apache-2.0 (Cachola et al., 2020).
- **External access:** the Hugging Face Hub only, to fetch the pinned `google-t5/t5-base` snapshot (~894 MB in total) at revision `a9723ea7f1b3…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'tokenizers==0.22.2',
    'sentencepiece==0.2.2',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 't5-base-text2text-pipeline',
    'repository_revision': '5d97bca80c1c599c9510efbe22ef801e0566442d',
    'embedded_module': 'src/t5_base_text2text_pipeline/pipeline.py',
    'embedded_modules': ['src/t5_base_text2text_pipeline/metrics.py', 'src/t5_base_text2text_pipeline/pipeline.py', 'src/t5_base_text2text_pipeline/samples.py'],
    'module_sha256': 'd409e1c15c9ac991ec1c6e5e93683ea927a0dc19281bc3050c51f9ba341822ac',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/t5_base_text2text_pipeline/` @ `5d97bca80c1c`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/t5_base_text2text_pipeline/metrics.py`

In [ ]:
"""Reference-based text-to-text metrics (ROUGE-1/2/L F1, own implementation) and the Lead baseline.

ROUGE follows the `rouge-score` package's recipe without stemming: lower-case, keep runs of ASCII letters
and digits as tokens, unigram/bigram overlap F1 for ROUGE-1/2, longest-common-subsequence F1 for ROUGE-L
(sentence-level, the whole output as one sequence). With several references per document the best score
over the references is taken, then averaged over documents and reported in percent.
Values are close to, but not identical with, `rouge-score` — no stemming, no bootstrap — and neither is a
human judgement of faithfulness.
"""

from __future__ import annotations

import re
from collections import Counter
from collections.abc import Mapping, Sequence
from typing import Any

_TOKEN_RE = re.compile(r"[a-z0-9]+")
_SENTENCE_RE = re.compile(r"(?<=[.!?])\s+")
METRIC_DEFINITIONS = {
    "rouge1": (
        "unigram overlap F1 between the output and the best-matching reference, averaged over documents; "
        "percent"
    ),
    "rouge2": (
        "bigram overlap F1 between the output and the best-matching reference, averaged over documents; "
        "percent"
    ),
    "rougeL": (
        "longest-common-subsequence F1 (whole output as one token sequence) against the best-matching "
        "reference, averaged over documents; percent"
    ),
    "tokenisation": (
        "lower-cased runs of ASCII letters and digits; no stemming; "
        "rouge-score-style, not rouge-score-identical"
    ),
}


def rouge_tokens(text: str) -> list[str]:
    return _TOKEN_RE.findall(text.lower())


def _f1(overlap: int, n_hyp: int, n_ref: int) -> float:
    if overlap == 0 or n_hyp == 0 or n_ref == 0:
        return 0.0
    precision, recall = overlap / n_hyp, overlap / n_ref
    return 2 * precision * recall / (precision + recall)


def rouge_n(hypothesis: str, reference: str, n: int) -> float:
    hyp = rouge_tokens(hypothesis)
    ref = rouge_tokens(reference)
    hyp_grams = Counter(tuple(hyp[i : i + n]) for i in range(len(hyp) - n + 1))
    ref_grams = Counter(tuple(ref[i : i + n]) for i in range(len(ref) - n + 1))
    overlap = sum((hyp_grams & ref_grams).values())
    return _f1(overlap, sum(hyp_grams.values()), sum(ref_grams.values()))


def _lcs_length(a: Sequence[str], b: Sequence[str]) -> int:
    if not a or not b:
        return 0
    previous = [0] * (len(b) + 1)
    for token in a:
        current = [0]
        for j, other in enumerate(b, start=1):
            current.append(previous[j - 1] + 1 if token == other else max(previous[j], current[j - 1]))
        previous = current
    return previous[-1]


def rouge_l(hypothesis: str, reference: str) -> float:
    hyp = rouge_tokens(hypothesis)
    ref = rouge_tokens(reference)
    return _f1(_lcs_length(hyp, ref), len(hyp), len(ref))


def rouge_scores(hypothesis: str, references: Sequence[str]) -> dict[str, float]:
    """Best ROUGE-1/2/L F1 over the references for one output (fractions in 0..1)."""
    if not references:
        raise ValueError("at least one reference is required")
    return {
        "rouge1": max(rouge_n(hypothesis, r, 1) for r in references),
        "rouge2": max(rouge_n(hypothesis, r, 2) for r in references),
        "rougeL": max(rouge_l(hypothesis, r) for r in references),
    }


def text_metrics(hypotheses: Sequence[str], references: Sequence[Sequence[str]]) -> dict[str, Any]:
    """Corpus ROUGE-1/2/L F1 in percent over parallel outputs and reference lists."""
    if len(hypotheses) != len(references):
        raise ValueError(f"{len(hypotheses)} outputs but {len(references)} reference lists")
    if not hypotheses:
        raise ValueError("no outputs to score")
    per_doc = [rouge_scores(h, r) for h, r in zip(hypotheses, references, strict=True)]
    return {
        "n": len(hypotheses),
        "rouge1": 100.0 * sum(d["rouge1"] for d in per_doc) / len(per_doc),
        "rouge2": 100.0 * sum(d["rouge2"] for d in per_doc) / len(per_doc),
        "rougeL": 100.0 * sum(d["rougeL"] for d in per_doc) / len(per_doc),
        "mean_output_words": sum(len(h.split()) for h in hypotheses) / len(hypotheses),
        "mean_reference_words": sum(len(r[0].split()) for r in references) / len(references),
        "definitions": dict(METRIC_DEFINITIONS),
    }


def lead_sentences(text: str, n_sentences: int = 1) -> str:
    """The first `n_sentences` sentences of a document (a naive `.!?` split)."""
    if n_sentences < 1:
        raise ValueError("n_sentences must be at least 1")
    return " ".join(_SENTENCE_RE.split(text.strip())[:n_sentences])


def lead_baseline(records: Sequence[Mapping[str, Any]], *, n_sentences: int = 1) -> dict[str, Any]:
    """Lead-N: the first N sentences of the source submitted as the output — the classic extractive floor."""
    result = text_metrics(
        [lead_sentences(r["source"], n_sentences) for r in records], [r["targets"] for r in records]
    )
    result["baseline"] = (
        f"lead-{n_sentences} (the first {n_sentences} sentence(s) of the source as the output)"
    )
    return result

**Module 2/3:** `src/t5_base_text2text_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""Text-to-text generation with the pinned ``google-t5/t5-base`` checkpoint.

The class loads weights only from a digest-verified local snapshot (``weights/t5-base/``) or, when
explicitly allowed, from the Hugging Face Hub at the pinned revision. The caller supplies the task
prefix (``summarize: ``, ``translate English to German: `` ...); this module never adds one at inference.

The adaptation contract (``evaluate``, ``adapt``, ``save_artifact``, ``from_artifact``) teaches a
caller-chosen prefix: it fine-tunes the last decoder blocks on a validated ``{id, source, targets}`` dataset
whose sources are prepended with that prefix, selects the epoch by validation ROUGE-L, and exports the
trained tensors as a safetensors adapter bound to the pinned base weights. The inference contract above is
unchanged by it.
"""

from __future__ import annotations

import hashlib
import json
import math
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

MODEL_ID = "google-t5/t5-base"
MODEL_REVISION = "a9723ea7f1b39c1eae772870f3b547bf6ef7e6c1"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "t5-base"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

MAX_INPUT_TOKENS = 512  # ``n_positions`` in the snapshot config.json; longer inputs are rejected, not cut
MAX_NEW_TOKENS = 512  # ceiling on decoder steps per call
DEFAULT_MAX_NEW_TOKENS = 64
MAX_TEXT_CHARS = 20_000  # pre-tokenisation guard on the input string
MAX_NUM_BEAMS = 8
DECISION_RULE = "greedy argmax per decoding step (num_beams=1); beam search when num_beams > 1; no sampling"
WEIGHT_FILE = "model.safetensors"
WEIGHT_SHA256 = (
    "a90903540cc02cbeb7ff9f823f1a80eb778c7e22426a0e620b01c77a5ec8f5b4"  # manifest digest of WEIGHT_FILE
)
PARAMETER_COUNT = 222_903_552
DECODER_LAYERS = 12  # config.json num_decoder_layers
DEFAULT_TRAINABLE_DECODER_LAYERS = 4  # the last four decoder blocks (37,757,952 parameters)
MAX_TRAIN_SOURCE_TOKENS = 512  # prefixed-source truncation ceiling during adaptation (never at inference)
MAX_TRAIN_TARGET_TOKENS = 64  # target truncation ceiling during adaptation
MAX_PREFIX_CHARS = 64
MAX_EVAL_RECORDS = 2_000
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.t5-base.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
# The four prefixes the upstream checkpoint was trained on, read from ``task_specific_params`` in the
# snapshot config.json. Reported back as ``known_prefix``; the pipeline does not prepend any of them.
TASK_PREFIXES = (
    "summarize: ",
    "translate English to German: ",
    "translate English to French: ",
    "translate English to Romanian: ",
)


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        return json.load(fh)


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one non-empty str that already carries its task prefix; the pipeline prepends none",
    "text_chars": [1, MAX_TEXT_CHARS],
    "input_tokens": [1, MAX_INPUT_TOKENS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "num_beams": [1, MAX_NUM_BEAMS],
    "task_prefixes": list(TASK_PREFIXES),
    "decision_rule": DECISION_RULE,
    "preprocessing": (
        "SentencePiece encoding with no prefix added and no truncation: an input over "
        "MAX_INPUT_TOKENS is rejected with a ValueError naming the count, never cut"
    ),
}


def _check_inputs(text: Any, max_new_tokens: Any, num_beams: Any) -> str:
    """Raise TypeError/ValueError naming the first violated ceiling; return the text.

    The encoder-token ceiling is not checked here because it needs the loaded tokenizer;
    ``_check_input_tokens`` applies it inside the pipeline once the count is known.
    """
    if not isinstance(text, str):
        raise TypeError(f"text must be str, got {type(text).__name__}")
    if not text.strip():
        raise ValueError("text is empty")
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(f"text has {len(text)} chars; ceiling is MAX_TEXT_CHARS={MAX_TEXT_CHARS}")
    for name, value, ceiling in (
        ("max_new_tokens", max_new_tokens, MAX_NEW_TOKENS),
        ("num_beams", num_beams, MAX_NUM_BEAMS),
    ):
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{name} must be an int")
        if not 1 <= value <= ceiling:
            raise ValueError(f"{name} must be between 1 and {ceiling}, got {value}")
    return text


def _check_input_tokens(n_input: int) -> int:
    """The encoder-token ceiling, applied once the tokenizer has counted."""
    if n_input > MAX_INPUT_TOKENS:
        raise ValueError(f"input is {n_input} tokens; ceiling is MAX_INPUT_TOKENS={MAX_INPUT_TOKENS}")
    return n_input


def _check_prefix(prefix: Any) -> str:
    """A task prefix is a short non-empty string ending in ': ' (the upstream convention)."""
    if not isinstance(prefix, str):
        raise TypeError("prefix must be str")
    if not prefix.strip():
        raise ValueError("prefix is empty")
    if len(prefix) > MAX_PREFIX_CHARS:
        raise ValueError(f"prefix has {len(prefix)} chars; ceiling is MAX_PREFIX_CHARS={MAX_PREFIX_CHARS}")
    if not prefix.endswith(": "):
        raise ValueError("prefix must end with ': ' like the trained prefixes (e.g. 'paper title: ')")
    return prefix


def known_prefix(text: str) -> str | None:
    """Which trained task prefix ``text`` starts with, or ``None``; nothing is prepended."""
    return next((prefix for prefix in TASK_PREFIXES if text.startswith(prefix)), None)


def validate_inputs(
    texts: Sequence[str],
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    num_beams: int = 1,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    Rejection is reported by raising exactly as ``generate`` would: both route through
    ``_check_inputs``. ``generate`` takes one text per call, so ``texts`` is the batch the notebook
    will loop over and every entry is validated with the same settings. An input that starts with
    no trained prefix is **not** rejected — the pipeline does not refuse it either — but the
    manifest records ``known_prefix: null`` so the caller can see it. The encoder-token ceiling
    (``MAX_INPUT_TOKENS``) needs the loaded tokenizer and is enforced inside ``generate``.
    """
    if isinstance(texts, str | bytes) or not isinstance(texts, Sequence):
        raise TypeError("texts must be a sequence of str, not a single string")
    if not texts:
        raise ValueError("texts must hold at least one item")
    checked = [_check_inputs(text, max_new_tokens, num_beams) for text in texts]
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per text")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[i] if names else f"input{i:02d}",
                "chars": len(text),
                "known_prefix": known_prefix(text),
            }
            for i, text in enumerate(checked)
        ],
        "max_new_tokens": max_new_tokens,
        "num_beams": num_beams,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], references: Sequence[str] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report for one ``generate`` result.

    With ``references`` (one or more reference outputs for the same input) the report carries the
    ROUGE-1/2/L F1 of that single output against its best-matching reference (`metrics.py`) with the
    verdict ``sample-sanity`` — one item is a plumbing check, not a quality measurement; the corpus-level
    stage is ``T5BaseText2TextPipeline.evaluate``. Without references the verdict is ``not-measurable``.
    """
    pass  # standalone rewrite (build_notebook.py): `from .metrics import rouge_scores` removed — names are kernel globals defined by the carried modules

    generation = result.get("generation", {})
    supplied = references is not None
    metrics: list[dict[str, Any]] = []
    if supplied:
        refs = [str(r) for r in references if str(r).strip()]
        if not refs:
            raise ValueError("references must hold at least one non-empty output")
        scores = rouge_scores(str(result.get("text", "")), refs)
        metrics = [
            {"id": name, "value": 100.0 * value, "estimation": "single item, best of the references"}
            for name, value in scores.items()
        ]
    return {
        "task": "caller-prefixed text-to-text generation (summarisation, translation, or a taught prefix)",
        "score_semantics": (
            "the pipeline emits no probability, confidence or score: generated_tokens, input_tokens "
            "and stopped_by are counts and flags, and "
            f"{generation.get('decision_rule', DECISION_RULE)} produces some token at every step "
            "with no minimum-probability cut-off and no shipped acceptance threshold; ROUGE, when "
            "references are supplied, is n-gram agreement with those references (own implementation), "
            "not faithfulness"
        ),
        "sample_kind": sample_kind,
        "n_generated_tokens": int(result.get("generated_tokens", 0)),
        "known_prefix": result.get("known_prefix"),
        "metrics": metrics,
        "baselines": [],
        "verdict": "sample-sanity" if supplied else "not-measurable",
        "reason": (
            "ROUGE-1/2/L are computed for one item against its reference outputs with the repository's own "
            "implementation; a single item states no dispersion and is not a quality measurement"
            if supplied
            else "no reference output was supplied, so ROUGE cannot be computed; "
            "a generation has no ground truth here"
        ),
        "needs": (
            "reference outputs from the deployment domain — a reference summary per document, a reference "
            "translation per sentence, a reference title per abstract — over enough items to state a "
            "dispersion, scored with `evaluate` (ROUGE-1/2/L, own implementation; BLEU/chrF for translation "
            "need the caller's own scorer), excluding or re-running outputs whose stopped_by is "
            "max_new_tokens; no proxy such as length ratio or copy rate substitutes for that"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class T5BaseText2TextPipeline:
    """``_runner(text, max_new_tokens, num_beams)`` -> ``(generated_text, generated_tokens, stopped_by)``;
    ``_count_tokens(text)`` -> encoder token count incl. EOS. Both injectable so tests run offline."""

    _runner: Callable[[str, int, int], tuple[str, int, str]]
    _count_tokens: Callable[[str], int]
    device: str = "cpu"
    source: str = "injected"
    adapter: dict[str, Any] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _tokenizer: Any = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> T5BaseText2TextPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, kwargs, source = str(root), dict(local_files_only=True), "local-snapshot"
        elif allow_download:
            location, kwargs, source = MODEL_ID, dict(revision=MODEL_REVISION), "hf-hub"
        else:
            raise FileNotFoundError(f"no verified snapshot at {root} and allow_download=False")
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import T5ForConditionalGeneration, T5TokenizerFast

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        tokenizer = T5TokenizerFast.from_pretrained(location, trust_remote_code=False, **kwargs)
        model = T5ForConditionalGeneration.from_pretrained(
            location, dtype=torch.float32, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()
        eos_id, pad_id = model.config.eos_token_id, model.config.pad_token_id

        def count_tokens(text: str) -> int:
            return len(tokenizer(text, truncation=False)["input_ids"])

        def runner(text: str, max_new_tokens: int, num_beams: int) -> tuple[str, int, str]:
            enc = tokenizer(text, return_tensors="pt", truncation=False).to(resolved_device)
            with torch.inference_mode():
                out = model.generate(
                    **enc, max_new_tokens=max_new_tokens, num_beams=num_beams, do_sample=False
                )
            ids = out[0].tolist()
            content = [t for t in ids if t not in (eos_id, pad_id)]
            stopped_by = "eos" if eos_id in ids else "max_new_tokens"
            return tokenizer.decode(content, skip_special_tokens=True), len(content), stopped_by

        return cls(runner, count_tokens, resolved_device, source, _model=model, _tokenizer=tokenizer)

    def _validate(self, text: Any, max_new_tokens: Any, num_beams: Any) -> int:
        text = _check_inputs(text, max_new_tokens, num_beams)
        return _check_input_tokens(self._count_tokens(text))

    def generate(
        self, text: str, *, max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS, num_beams: int = 1
    ) -> dict[str, Any]:
        """Run one prefixed input through encoder-decoder generation; the caller owns the task prefix."""
        n_input = self._validate(text, max_new_tokens, num_beams)
        generated, n_generated, stopped_by = self._runner(text, max_new_tokens, num_beams)
        if not isinstance(generated, str) or not isinstance(n_generated, int):
            raise RuntimeError("runner must return (str, int, str)")
        prefix = known_prefix(text)
        return {
            "text": generated,
            "generated_tokens": n_generated,
            "input_tokens": n_input,
            "stopped_by": stopped_by,
            "known_prefix": prefix,
            "generation": {
                "max_new_tokens": max_new_tokens,
                "num_beams": num_beams,
                "do_sample": False,
                "decision_rule": DECISION_RULE,
            },
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation -----------------------------------------------------------------------------------

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._tokenizer is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._tokenizer

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        prefix: str,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
        num_beams: int = 1,
    ) -> dict[str, Any]:
        """Generate from every record's prefixed source and score the outputs against its references
        (ROUGE-1/2/L). The prefix is the caller's; nothing checks it is one the model knows."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import text_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        prefix = _check_prefix(prefix)
        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        hypotheses, truncated = [], 0
        for record in checked:
            result = self.generate(
                prefix + record["source"], max_new_tokens=max_new_tokens, num_beams=num_beams
            )
            hypotheses.append(result["text"])
            truncated += result["stopped_by"] == "max_new_tokens"
        metrics = text_metrics(hypotheses, [r["targets"] for r in checked])
        metrics.update(
            {
                "prefix": prefix,
                "known_prefix": known_prefix(prefix),
                "hit_token_ceiling": truncated,
                "generation": {"max_new_tokens": max_new_tokens, "num_beams": num_beams, "do_sample": False},
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def _trainable_names(self, trainable_decoder_layers: int) -> list[str]:
        if (
            not isinstance(trainable_decoder_layers, int)
            or not 1 <= trainable_decoder_layers <= DECODER_LAYERS
        ):
            raise ValueError(f"trainable_decoder_layers must be an int in 1..{DECODER_LAYERS}")
        model, _ = self._require_model()
        first = DECODER_LAYERS - trainable_decoder_layers
        prefixes = tuple(f"decoder.block.{k}." for k in range(first, DECODER_LAYERS))
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        prefix: str,
        epochs: int = 2,
        lr: float = 5e-4,
        batch_size: int = 8,
        trainable_decoder_layers: int = DEFAULT_TRAINABLE_DECODER_LAYERS,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
        eval_generation: Mapping[str, Any] | None = None,
    ) -> dict[str, Any]:
        """Bounded supervised fine-tuning that teaches `prefix` on a validated text-to-text dataset.

        Every training source is prepended with `prefix`; only the last `trainable_decoder_layers` decoder
        blocks train (4 by default; the encoder, the shared embeddings, the earlier decoder blocks and the
        tied output projection stay frozen). Teacher-forced cross-entropy on the first reference, AdamW at
        a fixed learning rate with gradient clipping at 1.0, prefixed sources truncated to
        MAX_TRAIN_SOURCE_TOKENS and targets to MAX_TRAIN_TARGET_TOKENS **during training only**. Epoch 0
        records the frozen model's validation ROUGE under the same prefix; every epoch is scored on the
        validation split with `eval_generation` (the pipeline defaults unless given), and the epoch with the
        highest validation ROUGE-L is kept."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        prefix = _check_prefix(prefix)
        if not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-2):
            raise ValueError("lr must be in (0, 1e-2]")
        if not isinstance(batch_size, int) or not 1 <= batch_size <= 32:
            raise ValueError("batch_size must be an int in 1..32")
        names = self._trainable_names(trainable_decoder_layers)
        train_checked = validate_dataset(train)["records"]
        val_checked = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        )
        generation = dict(eval_generation or {})
        import torch

        torch.manual_seed(seed)
        model, tokenizer = self._require_model()
        started = time.perf_counter()
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        device = torch.device(self.device)

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            return {
                k: v
                for k, v in self.evaluate(val_checked, prefix=prefix, **generation).items()
                if k in ("rouge1", "rouge2", "rougeL", "n", "mean_output_words", "hit_token_ceiling")
            }

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress:
            progress(entry)
        best_rouge = entry["val"]["rougeL"] if entry["val"] else -math.inf
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        for epoch in range(1, epochs + 1):
            model.train()
            order = torch.randperm(len(train_checked), generator=generator).tolist()
            losses = []
            for start in range(0, len(order), batch_size):
                batch = [train_checked[i] for i in order[start : start + batch_size]]
                encoded = tokenizer(
                    [prefix + r["source"] for r in batch],
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=MAX_TRAIN_SOURCE_TOKENS,
                )
                labels = tokenizer(
                    text_target=[r["targets"][0] for r in batch],
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=MAX_TRAIN_TARGET_TOKENS,
                )["input_ids"]
                labels[labels == tokenizer.pad_token_id] = -100
                out = model(
                    input_ids=encoded["input_ids"].to(device),
                    attention_mask=encoded["attention_mask"].to(device),
                    labels=labels.to(device),
                )
                optimiser.zero_grad(set_to_none=True)
                out.loss.backward()
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                optimiser.step()
                losses.append(float(out.loss.detach()))
            model.eval()
            entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": score_val()}
            history.append(entry)
            if progress:
                progress(entry)
            current = entry["val"]["rougeL"] if entry["val"] else math.inf
            if current > best_rouge or not entry["val"]:
                best_rouge = current
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                best_epoch = epoch
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "prefix": prefix,
            "trainable_decoder_layers": trainable_decoder_layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "highest validation ROUGE-L" if val_checked else "final epoch (no validation split)",
            "lr": lr,
            "batch_size": batch_size,
            "max_train_source_tokens": MAX_TRAIN_SOURCE_TOKENS,
            "max_train_target_tokens": MAX_TRAIN_TARGET_TOKENS,
            "eval_generation": generation,
            "n_train": len(train_checked),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted decoder tensors as safetensors with a manifest naming the base and the prefix."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHT_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest and digest, then overwrite exactly the tensors it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        entry = manifest["files"][0]
        weights_path = root / entry["path"]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        model, _ = self._require_model()
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != manifest["tensors"]:
            raise ValueError("artifact tensor names differ from its manifest")
        state = model.state_dict()
        for key, value in tensors.items():
            if key not in state or not key.startswith("decoder.block."):
                raise ValueError(
                    f"artifact tensor {key} is not an adaptable decoder tensor of the base model"
                )
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key} has shape {tuple(value.shape)}, "
                    f"base has {tuple(state[key].shape)}"
                )
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        model.load_state_dict(merged, strict=True)
        model.eval()
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": manifest["tensors"],
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> T5BaseText2TextPipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 3/3:** `src/t5_base_text2text_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Text-to-text dataset contract for teaching a new task prefix: the pinned SciTLDR sample (abstract →
paper title), validation, seeded splitting, BYOD loaders and CSV export.

The default dataset is **real** and defines a task the pinned checkpoint was never trained on: SciTLDR
(Cachola et al., EMNLP Findings 2020; Apache-2.0) ships, for every computer-science paper, the abstract and
the paper's title. The three `SciTLDR-A` JSON-Lines files are fetched one by one from the project repository
at a pinned commit and refused on any byte-size or SHA-256 mismatch; the tutorial pairs each abstract with its
title under a **new prefix** (`paper title: `) that the caller chooses. The frozen model answers an unknown
prefix with whatever its trained prefixes make of it (the build record saw `False` and `True`), and the
fine-tuning question is whether a bounded adaptation of the last decoder blocks teaches the prefix on
held-out papers.

A record is ``{id, source, targets}``: the abstract (sentences joined by a space) without any prefix, and one
or more reference outputs — here the paper title. The prefix is supplied at evaluation and training time.
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_TEXT_CHARS, MODEL_ID` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "SciTLDR-A (abstract -> title)"
CORPUS_RELEASE = "allenai/scitldr @ 5ccad9c00a60ad75c9e04abf7f27d0f53f983b20"
CORPUS_BASE_URL = "https://raw.githubusercontent.com/allenai/scitldr/5ccad9c00a60ad75c9e04abf7f27d0f53f983b20/SciTLDR-Data/SciTLDR-A/"
CORPUS_FILES = {
    "train": ("train.jsonl", 3_155_015, "b222771d387be585cfdf5ae957b36757138415a352e0a3e3b23f73f87c3b1119"),
    "dev": ("dev.jsonl", 1_124_865, "3191fa98ccc09521332b7a1cd63b1930be4e8df125a235ccd31e40329709525e"),
    "test": ("test.jsonl", 1_204_107, "fb42dd6cd4f4a1928ae8a01a189456fbfe994a07e938bd49f68653933f6503c9"),
}
CORPUS_LICENSE = "Apache-2.0 (Cachola et al. 2020; allenai/scitldr)"
CORPUS_PAPERS = {"train": 1_992, "dev": 619, "test": 618}
DEFAULT_PREFIX = "paper title: "  # the new task prefix the tutorial teaches; not one of TASK_PREFIXES
DEFAULT_CACHE_DIR = Path("weights") / "scitldr"
MAX_SAMPLE_SOURCE_CHARS = 1_600  # abstracts above this are left out of the sample (the ceiling is 512 tokens)
MIN_SAMPLE_SOURCE_CHARS = 200
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 300, "validation": 50, "test": 100}
MIN_RECORDS = 8
MAX_RECORDS = 20_000
MAX_TARGET_CHARS = 400
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, bytes]:
    """Return the three pinned SciTLDR-A files (bytes) from the cache or the project repository, verified."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out = {}
    for split, (name, size, digest) in CORPUS_FILES.items():
        local = cache / name
        data = local.read_bytes() if local.is_file() else b""
        if len(data) != size or _sha256_bytes(data) != digest:
            url = CORPUS_BASE_URL + name
            if fetcher is not None:
                data = fetcher(url)
            else:
                with urllib.request.urlopen(url, timeout=120) as response:  # noqa: S310 (pinned https URL)
                    data = response.read()
            if len(data) != size or _sha256_bytes(data) != digest:
                raise ValueError(
                    f"{name}: fetched {len(data)} bytes with sha256 {_sha256_bytes(data)[:16]}…, "
                    f"pinned {size} / {digest[:16]}…"
                )
            local.write_bytes(data)
        out[split] = data
    return out


def read_corpus(files: Mapping[str, bytes]) -> dict[str, list[dict[str, Any]]]:
    """Parse the JSON-Lines members into abstract -> title records keeping each SciTLDR `paper_id`."""
    out = {}
    for split in CORPUS_FILES:
        if split not in files:
            raise ValueError(f"corpus is missing the {split} file")
        records = []
        for line in files[split].decode("utf-8").splitlines():
            if not line.strip():
                continue
            row = json.loads(line)
            title = str(row.get("title", "")).strip()
            records.append(
                {
                    "id": f"{split}-{row['paper_id']}",
                    "source": " ".join(str(s).strip() for s in row["source"]),
                    "targets": [title] if title else [],
                    "paper_id": str(row["paper_id"]),
                }
            )
        if len(records) != CORPUS_PAPERS[split]:
            raise ValueError(f"{split}: {len(records)} papers, expected {CORPUS_PAPERS[split]}")
        out[split] = records
    return out


def filter_records(records: Sequence[Mapping[str, Any]]) -> list[dict[str, Any]]:
    """Keep records whose source is within the sample length window and that have a non-empty title; drop
    repeated sources and repeated titles case-insensitively."""
    seen: set[str] = set()
    seen_titles: set[str] = set()
    kept = []
    for record in records:
        source = str(record["source"])
        if not MIN_SAMPLE_SOURCE_CHARS <= len(source) <= MAX_SAMPLE_SOURCE_CHARS or not record.get("targets"):
            continue
        key, title_key = source.lower(), str(record["targets"][0]).lower()
        if key in seen or title_key in seen_titles:
            continue
        seen.add(key)
        seen_titles.add(title_key)
        kept.append(dict(record))
    return kept


def build_sample_dataset(
    corpus: Mapping[str, Sequence[Mapping[str, Any]]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded draws from the three SciTLDR members: training from `train`, validation from `dev`, test from
    `test` — the release's own paper-disjoint partition, re-checked on abstracts by `check_split_disjoint`."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    source_of = {"train": "train", "validation": "dev", "test": "test"}
    rng = random.Random(seed)
    out: dict[str, list[dict[str, Any]]] = {}
    for name, size in sizes.items():
        pool = filter_records(corpus[source_of[name]])
        if size > len(pool):
            raise ValueError(f"requested {size} {name} records but only {len(pool)} fit")
        rng.shuffle(pool)
        out[name] = [
            {
                "id": f"{name}-{i:04d}",
                "source": r["source"],
                "targets": list(r["targets"]),
                "paper_id": r["paper_id"],
            }
            for i, r in enumerate(pool[:size])
        ]
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    return build_sample_dataset(
        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes
    )


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label} must be a mapping with id/source/targets")
    if "targets" not in record and "target" in record:
        record = {**record, "targets": [record["target"]]}
    for key in ("id", "source", "targets"):
        if key not in record:
            raise ValueError(f"{label} is missing {key!r}")
    rid, source, targets = record["id"], record["source"], record["targets"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label}: id must match {_ID_RE.pattern}")
    if not isinstance(source, str):
        raise ValueError(f"{label}: source must be a string")
    if not source.strip():
        raise ValueError(f"{label}: source is empty")
    if len(source) > MAX_TEXT_CHARS:
        raise ValueError(
            f"{label}: source has {len(source)} chars; ceiling is MAX_TEXT_CHARS={MAX_TEXT_CHARS}"
        )
    if isinstance(targets, str) or not isinstance(targets, Sequence) or not targets:
        raise ValueError(f"{label}: targets must be a non-empty list of reference outputs")
    checked_targets = []
    for j, target in enumerate(targets):
        if not isinstance(target, str) or not target.strip():
            raise ValueError(f"{label}: targets[{j}] must be a non-empty string")
        if len(target) > MAX_TARGET_CHARS:
            raise ValueError(
                f"{label}: targets[{j}] has {len(target)} chars; "
                f"ceiling is MAX_TARGET_CHARS={MAX_TARGET_CHARS}"
            )
        checked_targets.append(target.strip())
    item = {"id": rid, "source": source.strip(), "targets": checked_targets}
    if "paper_id" in record:
        item["paper_id"] = str(record["paper_id"])
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a text-to-text dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError("records must be a list of {id, source, targets} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = []
    ids: set[str] = set()
    sources: set[str] = set()
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        sources.add(item["source"].lower())
        checked.append(item)
    return {
        "records": checked,
        "n_records": len(checked),
        "unique_sources": len(sources),
        "references_per_record": {
            "min": min(len(r["targets"]) for r in checked),
            "max": max(len(r["targets"]) for r in checked),
        },
        "source_chars": {
            "min": min(len(r["source"]) for r in checked),
            "max": max(len(r["source"]) for r in checked),
        },
        "target_words": {
            "min": min(len(r["targets"][0].split()) for r in checked),
            "max": max(len(r["targets"][0].split()) for r in checked),
        },
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [[r["id"], r["source"], list(r["targets"])] for r in records]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no lower-cased source document appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = str(record["source"]).lower()
            if key in seen and seen[key] != name:
                raise ValueError(
                    f"a document ({record['source'][:60]!r}…) appears in both {seen[key]} and {name}"
                )
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train/validation/test after de-duplicating sources."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    unique = []
    for record in checked:
        key = record["source"].lower()
        if key not in seen:
            seen.add(key)
            unique.append(record)
    random.Random(seed).shuffle(unique)
    n_test = max(1, round(len(unique) * test_fraction))
    n_val = round(len(unique) * val_fraction)
    splits = {
        "test": unique[:n_test],
        "validation": unique[n_test : n_test + n_val],
        "train": unique[n_test + n_val :],
    }
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read records from a CSV (columns id, source, target — one reference per row), a JSON array or JSONL
    of ``{id, source, target}`` or ``{id, source, targets: [...]}`` objects (sources without any prefix)."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    suffix = file_path.suffix.lower()
    text = file_path.read_text(encoding="utf-8")
    if suffix == ".csv":
        rows = list(csv.DictReader(io.StringIO(text)))
        missing = {"id", "source", "target"} - set(rows[0].keys() if rows else set())
        if missing:
            raise ValueError(f"CSV is missing columns {sorted(missing)}")
        return [{"id": r["id"], "source": r["source"], "targets": [r["target"]]} for r in rows]
    if suffix == ".jsonl":
        return [_normalise(json.loads(line)) for line in text.splitlines() if line.strip()]
    if suffix == ".json":
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("JSON dataset must be an array of records")
        return [_normalise(r) for r in data]
    raise ValueError("BYOD datasets must be .csv, .json or .jsonl")


def _normalise(record: Any) -> Any:
    if isinstance(record, Mapping) and "targets" not in record and "target" in record:
        return {**{k: v for k, v in record.items() if k != "target"}, "targets": [record["target"]]}
    return record


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """One row per record with its first reference output."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["id", "source", "target"])
        writer.writeheader()
        for record in records:
            writer.writerow({"id": record["id"], "source": record["source"], "target": record["targets"][0]})
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `6`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `a9723ea7f1b3…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `T5BaseText2TextPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "t5-base",
  "modelId": "google-t5/t5-base",
  "revision": "a9723ea7f1b39c1eae772870f3b547bf6ef7e6c1",
  "files": [
    {
      "path": "README.md",
      "bytes": 8477,
      "sha256": "c43c23f205b6839a6adb10f0757265f76c1c6f96b7793a4e53582d3a61cbfb23"
    },
    {
      "path": "config.json",
      "bytes": 1208,
      "sha256": "46dd7cb62d29c81fb551e0ef1ea274c24a46ba441eeb948897706252933df033"
    },
    {
      "path": "generation_config.json",
      "bytes": 147,
      "sha256": "f5a1c7e2be8092018d8835128987edf0111637dd98e90599cc80310fef75d95a"
    },
    {
      "path": "model.safetensors",
      "bytes": 891646390,
      "sha256": "a90903540cc02cbeb7ff9f823f1a80eb778c7e22426a0e620b01c77a5ec8f5b4"
    },
    {
      "path": "spiece.model",
      "bytes": 791656,
      "sha256": "d60acb128cf7b7f2536e8f38a5b18a05535c9e14c7a355904270e15b0945ea86"
    },
    {
      "path": "tokenizer.json",
      "bytes": 1389353,
      "sha256": "d2acde0d8d71dd30a711834b07781b9c89feaac33fd332f60507699282740066"
    }
  ],
  "totalBytes": 893837231
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = T5BaseText2TextPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Referenced corpus, validation and split

`fetch_corpus` downloads the three pinned SciTLDR-A files (or reads them from the cache), refuses a byte-size or SHA-256 mismatch per file before it is parsed, and `read_corpus` flattens each JSON-Lines member into records whose `source` is the abstract's sentences joined by a space and whose single `targets` entry is the paper's title. `build_sample_dataset` keeps abstracts of 200..1,600 characters with a title, drops repeated abstracts and repeated titles, and draws 300 training records from the `train` member, 50 validation records from `dev` and 100 test records from `test` by a seeded shuffle — the release's own paper-disjoint partition. `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no abstract appears in two splits, and the training split is written to `outputs/t5_base_text2text_train.csv` in the shape BYOD expects. `PREFIX` is the task name every later cell prepends; the default is one the checkpoint has never seen.

Look for: 1,992 + 619 + 618 raw papers, three digests, splits 300 / 50 / 100, titles of 2..17 words, and four refusal probes — a duplicate id, an empty reference list, a missing field and a dataset too small to split — each rejected before `torch` does anything.

In [ ]:
import hashlib
import io
import json

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}
PREFIX = 'paper title: '  # @param {type:"string"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_papers = {'byod': len(records)}
else:
    corpus = read_corpus(fetch_corpus(cache_dir='weights/scitldr'))
    raw_papers = {name: len(part) for name, part in corpus.items()}
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} ({CORPUS_RELEASE}; {CORPUS_LICENSE})'
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
disjoint = check_split_disjoint(splits)
write_dataset_csv(train_records, 'outputs/t5_base_text2text_train.csv')
print({'data_source': data_source, 'prefix': PREFIX, 'prefix_is_trained': known_prefix(PREFIX) is not None, 'raw_papers': raw_papers, 'splits': disjoint, 'file_sha256': {k: v[2][:12] + '...' for k, v in CORPUS_FILES.items()}})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'unique_sources': manifest['unique_sources'], 'source_chars': manifest['source_chars'], 'target_words': manifest['target_words'], 'digest': manifest['digest'][:16] + '...'}})
print({'example': {'id': train_records[0]['id'], 'source': train_records[0]['source'][:200] + '...', 'targets': train_records[0]['targets']}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'empty reference list': [{**train_records[0], 'targets': []}, *train_records[1:8]],
    'missing field': [{'id': r['id'], 'source': r['source']} for r in train_records[:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Generate through the inference contract

Before any adaptation, the inference contract is exercised as it always was, on two prefixed inputs authored in this cell — a German translation and a summary, both under trained prefixes. `validate_inputs` applies exactly the checks `generate` applies (text type and character ceiling, `max_new_tokens` and `num_beams` within their ceilings) and returns an input manifest that records each input's `known_prefix`; the encoder-token ceiling `MAX_INPUT_TOKENS` (512) needs the real tokenizer and is enforced inside `generate`, which **rejects with a `ValueError` naming the count, never truncates**. An out-of-range `num_beams` is validated too and its rejection recorded as a finding. `generate` returns the text with `generated_tokens`, `input_tokens`, `stopped_by` (`eos`, or `max_new_tokens` when the output was cut) and `known_prefix`, and echoes the settings. **Score semantics:** the pipeline emits **no probability, confidence or score of any kind**. The third call uses the new prefix on one test abstract so the frozen model's answer to words it was never trained on is visible before any metric is read.

In [ ]:
import time

GEN_MAX_NEW_TOKENS = 24  # @param {type:"integer"}
NUM_BEAMS = 4  # @param {type:"integer"}

GEN = {'max_new_tokens': GEN_MAX_NEW_TOKENS, 'num_beams': NUM_BEAMS}
texts = ['translate English to German: The house is wonderful.', 'summarize: ' + test_records[0]['source']]
item_ids = [f'input{index:02d}' for index in range(len(texts))]
ceilings = {'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_INPUT_TOKENS': MAX_INPUT_TOKENS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'MAX_NUM_BEAMS': MAX_NUM_BEAMS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'MAX_PREFIX_CHARS': MAX_PREFIX_CHARS}
print(ceilings)
print({'decision_rule': DECISION_RULE, 'task_prefixes': list(TASK_PREFIXES)})
input_manifest = validate_inputs(texts, max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=NUM_BEAMS, names=item_ids)
try:
    validate_inputs(texts, max_new_tokens=GEN_MAX_NEW_TOKENS, num_beams=MAX_NUM_BEAMS + 1)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'num-beams-ceiling-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/t5_base_text2text_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
results = []
for item_id, text in zip(item_ids, texts, strict=True):
    started = time.perf_counter()
    result = pipe.generate(text, **GEN)
    results.append({'id': item_id, 'input': text, 'seconds': round(time.perf_counter() - started, 3), **result})
    print({'id': item_id, 'known_prefix': result['known_prefix'], 'input_tokens': result['input_tokens'], 'generated_tokens': result['generated_tokens'], 'stopped_by': result['stopped_by'], 'seconds': results[-1]['seconds'], 'text': result['text']})
checks = {
    'one_result_per_input': len(results) == len(texts),
    'generated_within_ceiling': all(r['generated_tokens'] <= GEN_MAX_NEW_TOKENS for r in results),
    'input_within_ceiling': all(r['input_tokens'] <= MAX_INPUT_TOKENS for r in results),
    'every_input_has_known_prefix': all(r['known_prefix'] is not None for r in results),
    'settings_echoed': all(r['generation']['max_new_tokens'] == GEN_MAX_NEW_TOKENS and r['generation']['num_beams'] == NUM_BEAMS and r['generation']['do_sample'] is False for r in results),
}
if not all(checks.values()):
    raise RuntimeError(f'generate output failed a sanity check: {checks}')
new_prefix_probe = pipe.generate(PREFIX + test_records[0]['source'], **GEN)
print({'checks': checks, 'findings': len(input_manifest['findings']), 'no_score': 'the pipeline emits no probability or quality score'})
print({'frozen_model_under_new_prefix': {'prefix': PREFIX, 'known_prefix': new_prefix_probe['known_prefix'], 'text': new_prefix_probe['text'], 'reference_title': test_records[0]['targets'][0]}})

## 6. Baselines and the frozen model's score on the test split

Three numbers frame the adaptation, all under the settings of Section 5. The **Lead-1 baseline** submits the first sentence of each abstract as its title: what a system that does no modelling gets. The **frozen model under the new prefix** generates from `paper title: ` + abstract — an unknown prefix is just words, so expect a near-zero score and one-word outputs. The **frozen model under `summarize: `**, its closest trained task, is scored against the same titles as a second reference point: a news-style summary of the abstract overlaps the title more than nothing, but it is not a title. All three use corpus **ROUGE-1**, **ROUGE-2** and **ROUGE-L** F1 (rouge-score-style, not rouge-score-identical); read `mean_output_words` beside every score. About a minute and a half on CPU.

In [ ]:
baseline_lead1 = lead_baseline(test_records, n_sentences=1)
print({'lead1_baseline': {'rouge1': round(baseline_lead1['rouge1'], 2), 'rouge2': round(baseline_lead1['rouge2'], 2), 'rougeL': round(baseline_lead1['rougeL'], 2), 'mean_output_words': round(baseline_lead1['mean_output_words'], 1), 'n': baseline_lead1['n']}})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, prefix=PREFIX, **GEN)
print({'frozen_model_new_prefix_test': {'rouge1': round(frozen_test['rouge1'], 2), 'rouge2': round(frozen_test['rouge2'], 2), 'rougeL': round(frozen_test['rougeL'], 2), 'mean_output_words': round(frozen_test['mean_output_words'], 1), 'mean_reference_words': round(frozen_test['mean_reference_words'], 1), 'hit_token_ceiling': frozen_test['hit_token_ceiling'], 'known_prefix': frozen_test['known_prefix'], 'n': frozen_test['n'], 'verdict': frozen_test['verdict']}, 'seconds': round(time.perf_counter() - t0, 1)})
t0 = time.perf_counter()
frozen_summarize = pipe.evaluate(test_records, prefix='summarize: ', **GEN)
print({'frozen_model_summarize_test': {'rouge1': round(frozen_summarize['rouge1'], 2), 'rouge2': round(frozen_summarize['rouge2'], 2), 'rougeL': round(frozen_summarize['rougeL'], 2), 'mean_output_words': round(frozen_summarize['mean_output_words'], 1), 'known_prefix': frozen_summarize['known_prefix']}, 'seconds': round(time.perf_counter() - t0, 1)})
print({'definitions': frozen_test['definitions']})
for record in test_records[:2]:
    print({'frozen_new_prefix': pipe.generate(PREFIX + record['source'], **GEN)['text'], 'frozen_summarize': pipe.generate('summarize: ' + record['source'], **GEN)['text'], 'reference': record['targets'][0]})
assert baseline_lead1['rouge1'] > 0.0

## 7. Bounded fine-tuning that teaches the prefix

`pipe.adapt` prepends `PREFIX` to every training abstract and trains only the last `TRAINABLE_DECODER_LAYERS` decoder blocks — four by default, 37,757,952 of 222,903,552 parameters; the encoder, the shared embeddings, the tied output projection and the earlier decoder blocks stay frozen — with teacher-forced cross-entropy on the title, AdamW at a fixed learning rate, gradient clipping at 1.0, seeded shuffling and no scheduler. Prefixed sources are truncated to 512 and targets to 64 BPE tokens **during training only**. Epoch 0 records the frozen model's validation ROUGE under the same prefix and settings; every epoch is scored the same way, and the epoch with the highest validation ROUGE-L is kept.

Watch validation ROUGE-L go from near zero to the thirties and `mean_output_words` settle near title length within two epochs (about 100 s of training plus a validation pass per epoch on CPU). The build record's counter-examples: two decoder blocks at a lower learning rate reached ROUGE-L 16 in two epochs and still produced `False` for some abstracts — the new prefix needs more capacity than an in-domain tweak.

In [ ]:
EPOCHS = 2  # @param {type:"integer"}
LEARNING_RATE = 5e-4  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}
TRAINABLE_DECODER_LAYERS = 4  # @param {type:"integer"}

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row['val_rouge1'] = round(entry['val']['rouge1'], 2)
        row['val_rouge2'] = round(entry['val']['rouge2'], 2)
        row['val_rougeL'] = round(entry['val']['rougeL'], 2)
        row['val_mean_output_words'] = round(entry['val']['mean_output_words'], 1)
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, prefix=PREFIX, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_decoder_layers=TRAINABLE_DECODER_LAYERS, eval_generation=GEN, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'prefix': adapt_result['prefix'], 'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test split was never used for training or epoch selection, and no abstract in it appears in the training or validation splits. The adapted model is scored under the new prefix exactly as the frozen model was in Section 6, and the four numbers are put side by side. Look for a ROUGE-L in the high twenties or thirties — above the Lead-1 baseline and above the trained `summarize: ` prefix — with `mean_output_words` near the reference length; the cell asserts the adapted ROUGE-L is above the frozen ROUGE-L under the same prefix. One hundred abstracts from one seeded split of one corpus give no dispersion estimate; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a taught prefix on paper abstracts says nothing about your task until you measure it there. The adapter changes only the last decoder blocks, so the trained prefixes are affected too; Section 9 checks the German translation once more after adaptation.

In [ ]:
adapted_test = pipe.evaluate(test_records, prefix=PREFIX, **GEN)
adapted_val = pipe.evaluate(val_records, prefix=PREFIX, **GEN)
comparison = {
    metric: {'lead1': round(baseline_lead1[metric], 2), 'frozen_summarize': round(frozen_summarize[metric], 2), 'frozen_new_prefix': round(frozen_test[metric], 2), 'adapted': round(adapted_test[metric], 2)}
    for metric in ('rouge1', 'rouge2', 'rougeL', 'mean_output_words')
}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 2) for metric in ('rouge1', 'rouge2', 'rougeL')}
for metric, row in comparison.items():
    print({metric: row})
for record in test_records[:2]:
    print({'adapted': pipe.generate(PREFIX + record['source'], **GEN)['text'], 'reference': record['targets'][0]})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'prefix': PREFIX,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'generation': frozen_test['generation'],
    'baselines': {'lead1': baseline_lead1, 'frozen_summarize_prefix': frozen_summarize},
    'frozen_test': frozen_test,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/t5_base_text2text_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['rougeL'] > frozen_test['rougeL']
print({'report': 'outputs/t5_base_text2text_evaluation_report.json'})

## 9. Generate for new abstracts, export the adapter and reload it

Four abstracts that were in none of the splits are given titles by the adapted model through the same `generate` contract as Section 5 and scored with `pipe.evaluate` (a `measured-small-sample` verdict, because four documents carry no dispersion estimate); the single-document `evaluation_report` helper — the inference-stage helper, which now scores supplied references as `sample-sanity` — is written for the first of them, and the German translation from Section 5 is generated once more so the effect of the adapter on a trained prefix is visible.

`pipe.save_artifact` writes the trained tensors — the last four decoder blocks, about 151 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the prefix it was trained for, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `T5BaseText2TextPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digest **before** deserialising, refuses any tensor that is not an adaptable decoder tensor, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical outputs (VER4).

In [ ]:
import csv
import shutil

if USE_BYOD:
    new_records = [{**r, 'id': f'new-{i:02d}'} for i, r in enumerate(test_records[:4])]
else:
    used = {r['source'].lower() for part in splits.values() for r in part}
    new_records = [{**r, 'id': f'new-{i:02d}'} for i, r in enumerate([r for r in filter_records(corpus['dev']) if r['source'].lower() not in used][:4])]
new_metrics = pipe.evaluate(new_records, prefix=PREFIX, **GEN)
new_results = []
for record in new_records:
    item = pipe.generate(PREFIX + record['source'], **GEN)
    new_results.append({'id': record['id'], 'output': item['text'], 'reference': record['targets'][0], 'generated_tokens': item['generated_tokens'], 'input_tokens': item['input_tokens'], 'stopped_by': item['stopped_by']})
    print({k: new_results[-1][k] for k in ('id', 'output', 'reference')})
single_report = evaluation_report(pipe.generate(PREFIX + new_records[0]['source'], **GEN), new_records[0]['targets'], sample_kind='one unseen SciTLDR abstract' if not USE_BYOD else 'one BYOD test record')
german_after = pipe.generate(texts[0], **GEN)
print({'new_abstracts': {'n': new_metrics['n'], 'rouge1': round(new_metrics['rouge1'], 2), 'rougeL': round(new_metrics['rougeL'], 2), 'verdict': new_metrics['verdict']}, 'single_document_report_verdict': single_report['verdict'], 'german_translation_before_after': [results[0]['text'], german_after['text']]})
with open('outputs/t5_base_text2text_generations.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(new_results[0]))
    writer.writeheader()
    writer.writerows(new_results)

artifact_dir = Path('outputs/t5_base_text2text_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 't5_base_text2text', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'prefix': artifact_manifest['adapter']['prefix'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = T5BaseText2TextPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = [pipe.generate(PREFIX + r['source'], **GEN)['text'] for r in test_records[:6]]
after = [reloaded.generate(PREFIX + r['source'], **GEN)['text'] for r in test_records[:6]]
parity = {'identical_outputs': sum(a == b for a, b in zip(before, after, strict=True)), 'of': len(before)}
print({'reload_parity': parity, 'reloaded_prefix': reloaded.adapter['prefix'], 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_outputs'] == parity['of']

weight_entry = next(entry for entry in snapshot['files'] if entry['path'] == WEIGHT_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': weight_entry['sha256']},
    'data_source': data_source,
    'prefix': PREFIX,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'base_url': CORPUS_BASE_URL, 'files': {k: {'name': v[0], 'bytes': v[1], 'sha256': v[2]} for k, v in CORPUS_FILES.items()}, 'license': CORPUS_LICENSE},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'items': [{k: r[k] for k in ('id', 'input', 'text', 'known_prefix', 'input_tokens', 'generated_tokens', 'stopped_by', 'seconds')} for r in results], 'frozen_new_prefix_probe': new_prefix_probe['text'], 'german_after_adaptation': german_after['text']},
    'comparison': comparison,
    'new_abstracts': new_metrics,
    'single_document_report': single_report,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'float32', 'source': pipe.source},
}
with open('outputs/t5_base_text2text_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

An unknown prefix means nothing to the frozen model — `paper title: ` produced one-word answers scoring near zero, while its trained `summarize: ` prefix overlapped the titles about as much as the abstract's first sentence — and a bounded fine-tuning of the last four decoder blocks on 300 abstract–title pairs teaches the prefix in a few minutes on CPU, lifting held-out ROUGE-L above both reference points with outputs at title length, with a 151 MB adapter that reloads to identical outputs. That is the claim: the adaptation contract can teach a new task prefix end to end on a real referenced corpus, and the numbers it produces are read against a Lead baseline and the frozen model's nearest trained task rather than in isolation.

The test split is 100 abstracts from one seeded split of one corpus, the metrics are three n-gram overlap scores (own implementation, not rouge-score-identical, and none a judgement of whether a title is good), and titles are short and formulaic. So a gain here says the contract works, not that the adapted model writes good titles for your papers, that it handles long or technical inputs, or that its outputs are faithful. The adapter changes the last decoder blocks, which every prefix shares, so the trained tasks are affected too — Section 9 shows the German translation before and after — and nothing here measures that beyond one sentence.

Three things to carry to real data. **References first:** the Lead baseline and the frozen model under its nearest trained prefix on *your* references are the numbers to read before any adapted one. **Leakage:** de-duplicate sources across splits (the contract does this case-insensitively) and split by document collection or author when your pairs come from one. **Ceilings:** prefixed inputs over `MAX_INPUT_TOKENS` are refused at inference and truncated to 512 tokens only during training — long-document tasks are out of scope.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real referenced corpus, validate the demonstrated dataset contract without leakage, execute the inference contract and a bounded fine-tuning that teaches a new prefix, evaluate against a trivial baseline and the frozen model on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, output quality or faithfulness on any other task, a usable acceptance threshold, or production fitness.

**Optional experiments (they do not affect the default path):** set `TRAINABLE_DECODER_LAYERS = 2` and watch the prefix fail to take in two epochs; set `PREFIX = 'summarize: '` to re-target a trained prefix to titles and compare the frozen and adapted scores; set `NUM_BEAMS = 1` and read the greedy scores; or bring your own input–output pairs and prefix through BYOD and read the Lead baseline before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/t5-base-text2text-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/t5-base-text2text-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/t5-base-text2text-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/google-t5/t5-base
- Upstream code: https://github.com/google-research/text-to-text-transfer-transformer
- Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer (Raffel et al., JMLR 2020): https://arxiv.org/abs/1910.10683
- TLDR: Extreme Summarization of Scientific Documents (Cachola et al., EMNLP Findings 2020; SciTLDR, Apache-2.0): https://arxiv.org/abs/2004.15011
- ROUGE: A Package for Automatic Evaluation of Summaries (Lin, 2004): https://aclanthology.org/W04-1013
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)